# sol01: Tool-Using Support Agent

Contains:
- the same scenario as `01_mock`
- one complete reference implementation
- grading tests


In [ ]:
import inspect
import json
from copy import deepcopy
from typing import Any, Callable

ORDERS_DB = {
    "u-100": [
        {"order_id": "o-900", "status": "delivered", "days_since_delivery": 3, "amount": 42.5}
    ],
    "u-200": [
        {"order_id": "o-901", "status": "in_transit", "days_since_delivery": 0, "amount": 81.0}
    ],
}
REFUNDS: list[dict[str, Any]] = []


def reset_state() -> None:
    REFUNDS.clear()


def get_orders(user_id: str) -> list[dict[str, Any]]:
    if user_id == "boom":
        raise RuntimeError("orders backend unavailable")
    return deepcopy(ORDERS_DB.get(user_id, []))


def policy_check(order_id: str, reason: str, days_since_delivery: int) -> dict[str, Any]:
    eligible = reason.lower() in {"damaged", "wrong_item"} and days_since_delivery <= 14
    return {
        "order_id": order_id,
        "eligible": eligible,
        "policy_reason": "allowed" if eligible else "outside_policy",
    }


def create_refund(order_id: str, amount: float) -> dict[str, Any]:
    if amount <= 0:
        raise ValueError("refund amount must be positive")
    record = {"order_id": order_id, "amount": amount, "status": "submitted"}
    REFUNDS.append(record)
    return deepcopy(record)


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_orders": get_orders,
    "policy_check": policy_check,
    "create_refund": create_refund,
}


class ScriptedModel:
    def __init__(self, responses: list[dict[str, Any]]) -> None:
        self._responses = deepcopy(responses)
        self._index = 0

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        if self._index >= len(self._responses):
            return {"stop_reason": "end_turn", "output_text": "No scripted response left."}
        response = self._responses[self._index]
        self._index += 1
        return deepcopy(response)


In [ ]:
def validate_tool_call(tool_call: dict[str, Any], tool_registry: dict[str, Callable[..., Any]]) -> str | None:
    required = {"id", "name", "input"}
    if not required.issubset(tool_call):
        return "tool_call_missing_required_fields"

    name = tool_call["name"]
    payload = tool_call["input"]
    if name not in tool_registry:
        return "unknown_tool"
    if not isinstance(payload, dict):
        return "tool_input_must_be_object"

    sig = inspect.signature(tool_registry[name])
    missing: list[str] = []
    for param in sig.parameters.values():
        if param.default is inspect._empty and param.name not in payload:
            missing.append(param.name)
    if missing:
        return f"missing_required_args:{','.join(sorted(missing))}"

    return None


def execute_tool_call(tool_call: dict[str, Any], tool_registry: dict[str, Callable[..., Any]]) -> dict[str, Any]:
    tool_id = str(tool_call.get("id", "missing_id"))
    tool_name = str(tool_call.get("name", "missing_name"))

    validation_error = validate_tool_call(tool_call, tool_registry)
    if validation_error:
        return {
            "role": "tool",
            "tool_call_id": tool_id,
            "name": tool_name,
            "is_error": True,
            "content": json.dumps({"error": validation_error}, sort_keys=True),
        }

    try:
        result = tool_registry[tool_name](**tool_call["input"])
        return {
            "role": "tool",
            "tool_call_id": tool_id,
            "name": tool_name,
            "is_error": False,
            "content": json.dumps({"result": result}, sort_keys=True),
        }
    except Exception as exc:  # pragma: no cover - explicit for interview robustness
        return {
            "role": "tool",
            "tool_call_id": tool_id,
            "name": tool_name,
            "is_error": True,
            "content": json.dumps({"error": str(exc)}, sort_keys=True),
        }


def run_agent(
    user_prompt: str,
    model: Callable[[list[dict[str, Any]]], dict[str, Any]],
    tool_registry: dict[str, Callable[..., Any]],
    max_steps: int = 6,
) -> dict[str, Any]:
    messages: list[dict[str, Any]] = [{"role": "user", "content": user_prompt}]

    for _ in range(max_steps):
        response = model(messages)
        stop_reason = response.get("stop_reason")

        if stop_reason == "tool_use":
            tool_calls = response.get("tool_calls", [])
            if not isinstance(tool_calls, list):
                raise RuntimeError("tool_calls_must_be_list")
            for tool_call in tool_calls:
                messages.append(execute_tool_call(tool_call, tool_registry))
            continue

        if stop_reason == "end_turn":
            return {"final_text": str(response.get("output_text", "")).strip(), "messages": messages}

        raise RuntimeError(f"unsupported_stop_reason:{stop_reason}")

    raise RuntimeError("max_steps_exceeded")


In [ ]:
def _tool_messages(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [m for m in messages if m.get("role") == "tool"]


def run_exam01_tests() -> None:
    reset_state()

    # 1) Single tool call
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "tool_calls": [{"id": "t1", "name": "get_orders", "input": {"user_id": "u-100"}}],
            },
            {"stop_reason": "end_turn", "output_text": "Order o-900 is delivered."},
        ]
    )
    result = run_agent("Where is my order?", model, TOOL_REGISTRY)
    assert "delivered" in result["final_text"].lower()
    assert len(_tool_messages(result["messages"])) == 1

    # 2) Multiple tools in one model turn
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "tool_calls": [
                    {
                        "id": "t2",
                        "name": "policy_check",
                        "input": {"order_id": "o-900", "reason": "damaged", "days_since_delivery": 3},
                    },
                    {"id": "t3", "name": "create_refund", "input": {"order_id": "o-900", "amount": 42.5}},
                ],
            },
            {"stop_reason": "end_turn", "output_text": "Refund submitted."},
        ]
    )
    result = run_agent("Refund my damaged item", model, TOOL_REGISTRY)
    assert len(_tool_messages(result["messages"])) == 2
    assert REFUNDS and REFUNDS[-1]["order_id"] == "o-900"

    # 3) Missing args -> is_error tool message
    model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "tool_calls": [{"id": "bad-args", "name": "get_orders", "input": {}}]},
            {"stop_reason": "end_turn", "output_text": "Handled error."},
        ]
    )
    result = run_agent("debug", model, TOOL_REGISTRY)
    tool_msg = _tool_messages(result["messages"])[0]
    assert tool_msg["is_error"] is True
    assert "missing_required_args" in tool_msg["content"]

    # 4) Runtime exception -> is_error tool message
    model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "tool_calls": [{"id": "boom", "name": "get_orders", "input": {"user_id": "boom"}}]},
            {"stop_reason": "end_turn", "output_text": "Handled exception."},
        ]
    )
    result = run_agent("debug", model, TOOL_REGISTRY)
    tool_msg = _tool_messages(result["messages"])[0]
    assert tool_msg["is_error"] is True
    assert "backend unavailable" in tool_msg["content"]

    # 5) Max step protection
    model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "tool_calls": [{"id": "loop1", "name": "get_orders", "input": {"user_id": "u-200"}}]},
            {"stop_reason": "tool_use", "tool_calls": [{"id": "loop2", "name": "get_orders", "input": {"user_id": "u-200"}}]},
            {"stop_reason": "tool_use", "tool_calls": [{"id": "loop3", "name": "get_orders", "input": {"user_id": "u-200"}}]},
        ]
    )
    try:
        run_agent("loop", model, TOOL_REGISTRY, max_steps=2)
        raise AssertionError("Expected max_steps_exceeded")
    except RuntimeError as exc:
        assert "max_steps_exceeded" in str(exc)

    print("01_mock tests passed")


run_exam01_tests()


## Walkthrough: Exactly How to Solve `01_mock`

### 0) First 2 minutes (do this before coding)
- Read function TODOs and write this mini-plan in comments:
  1. `validate_tool_call`
  2. `execute_tool_call`
  3. `run_agent`
- Do **not** start `run_agent` first.

### 1) Should I read tests now?
Yes, but fast:
- Spend 3-4 minutes scanning test names and assertions only.
- Extract contracts from tests:
  - Single and multiple tool calls must work.
  - Missing args and runtime failures must return `is_error=True`.
  - `max_steps` must raise `RuntimeError("max_steps_exceeded")`.
- Then stop reading tests and implement TODOs.

### 2) Coding order with checkpoints
1. `validate_tool_call`:
   - check required keys (`id`, `name`, `input`)
   - check tool exists
   - check `input` is dict
   - check required args from function signature
2. `execute_tool_call`:
   - call validator first
   - on validation/runtime error return tool message with `is_error=True`
   - on success return tool message with JSON result payload
3. `run_agent`:
   - initialize `messages=[{"role":"user", ...}]`
   - loop up to `max_steps`
   - if `tool_use`: execute **all** tool calls, append messages, continue
   - if `end_turn`: return final text + messages
   - else: unsupported stop reason error

### 3) One concrete example to narrate aloud
Use test case #2 (multiple tools):
- Model returns `policy_check` and `create_refund` in one turn.
- You execute both and append two tool messages.
- Next model turn ends with "Refund submitted."
- Why this matters: proves your loop handles batched tool calls, not only one.

### 4) What to say while coding (verbatim-safe)
- "I scanned tests first to lock the contract, now I am implementing TODOs in dependency order."
- "I am enforcing a loop invariant: every `tool_use` turn appends tool outputs before next model call."
- "I am returning structured tool errors instead of crashing so the conversation can recover."
- "After baseline passes, I check failure paths: missing args, runtime exception, and max-step loop safety."

### 5) Self-check questions before final run
- Do I process all tool calls in a turn?
- Can unknown tools and missing args fail safely?
- Is `max_steps` guaranteed to stop infinite loops?
- Are error messages JSON and debuggable?
